# Embeddings

## Load Secrets

In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

## Convert PDF to Markdown

In [3]:
import pymupdf4llm

PATH_TO_ARTICLE = "./data/article.pdf"

md_text = pymupdf4llm.to_markdown(PATH_TO_ARTICLE)
print(f"Got {len(md_text):,} characters of markdown.\n")
print(md_text[:500], "...")

Got 34,152 characters of markdown.

## **Hidden Technical Debt in Agentic Systems** 

Issue #02 — Eleven years later, the same diagram is being redrawn 

**==> picture [28 x 28] intentionally omitted <==**

MIGUEL OTERO PEDRIDO MAY 06, 2026 

**==> picture [414 x 413] intentionally omitted <==**

Hey friends! 👋 

## Welcome to **Issue #02** of **The AI Systems Engineer Journey** . 

Last week we drew the big map: AI Engineering is a _superset_ of ML Engineering — same chassis �FTI�, different organs. We walked the LinkedIn job-tit ...


## Chunk the Markdown Text

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_text(md_text)
sizes = [len(c) for c in chunks]
print(f"{len(chunks)} chunks. Size: min {min(sizes)}, max {max(sizes)}, median {sorted(sizes)[len(sizes) // 2]}.")
print(f"\nFirst Chunk Preview:\n{chunks[0][:200]}...")

91 chunks. Size: min 135, max 509, median 411.

First Chunk Preview:
## **Hidden Technical Debt in Agentic Systems** 

Issue #02 — Eleven years later, the same diagram is being redrawn 

**==> picture [28 x 28] intentionally omitted <==**

MIGUEL OTERO PEDRIDO MAY 06, ...


## Embed The Chunks (Dense Embedding)

In [5]:
import os
import cohere

co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])

dense = co.embed(
    model="embed-v4.0",
    input_type="search_document",
    embedding_types=["float"],
    texts=chunks,
).embeddings.float


print(f"Embedded {len(dense)} chunks.")
print(f"Each vector has {len(dense[0])} numbers. First 5: {dense[0][:5]}")

Embedded 91 chunks.
Each vector has 1536 numbers. First 5: [0.015113866, -0.041693423, 0.0172854, -0.0005591696, 0.0086427]


## Sanity Check for Similarity

In [6]:
import math


def cosine(a, b):
    dot = sum([x * y for x, y in zip(a, b)])
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return dot / (na * nb)


mid = len(chunks) // 2
print(
    f"cos(chunks[0], chunks[1])       = {cosine(dense[0], dense[1]):.3f}   ← adjacent"
)
print(
    f"cos(chunks[0], chunks[{mid:>3}])     = {cosine(dense[0], dense[mid]):.3f}   ← different part of doc"
)
print(
    f"cos(chunks[0], chunks[-1])      = {cosine(dense[0], dense[-1]):.3f}   ← far end of doc"
)

cos(chunks[0], chunks[1])       = 0.547   ← adjacent
cos(chunks[0], chunks[ 45])     = 0.304   ← different part of doc
cos(chunks[0], chunks[-1])      = 0.400   ← far end of doc
